# Serve Qwen2.5 3B Instruct with vLLM on Colab

This notebook starts an OpenAI-compatible vLLM server for `Qwen/Qwen2.5-3B-Instruct` on Colab port `8000`, exposes that port with ngrok, and prints the `LLM_API_URL` settings needed by the Legal AI Assistant backend.

Requirements:
- Colab GPU runtime, preferably T4 or better.
- Public Hugging Face access to `Qwen/Qwen2.5-3B-Instruct`.
- An ngrok auth token.

Do not hardcode real tokens in this notebook. Use Colab Secrets or enter them when prompted.

## 1. Check GPU Runtime

In Colab, choose `Runtime -> Change runtime type -> T4 GPU` before running the notebook.

In [ ]:
!nvidia-smi

import shutil

if shutil.which("nvidia-smi") is None:
    raise RuntimeError("No NVIDIA GPU detected. Enable a GPU runtime before continuing.")

## 2. Install Runtime Dependencies

This installs vLLM, Hugging Face Hub tooling, ngrok support, and HTTP clients used for endpoint tests. If Colab reports package conflicts, restart the runtime once and rerun the notebook from the top.

In [ ]:
!pip install -q -U vllm huggingface_hub pyngrok openai requests
!vllm --version

## 3. Configure Model and Server Settings

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
PORT = 8000
API_KEY = "dummy-key"

# Keep context modest on free T4 to reduce KV-cache memory pressure.
MAX_MODEL_LEN = 4096
GPU_MEMORY_UTILIZATION = 0.90

print("Model:", MODEL_NAME)
print("Local vLLM endpoint:", f"http://127.0.0.1:{PORT}/v1")

## 4. Authenticate ngrok

Recommended Colab Secret name:
- `NGROK_AUTHTOKEN`

Qwen/Qwen2.5-3B-Instruct is public on Hugging Face, so this notebook does not require a Hugging Face token.

In [ ]:
import getpass
import os

from pyngrok import ngrok

def get_secret(name: str, prompt: str) -> str:
    value = os.environ.get(name)
    if value:
        return value
    try:
        from google.colab import userdata
        value = userdata.get(name)
        if value:
            return value
    except Exception:
        pass
    return getpass.getpass(prompt)

NGROK_AUTHTOKEN = get_secret("NGROK_AUTHTOKEN", "Enter ngrok auth token: ")

ngrok.set_auth_token(NGROK_AUTHTOKEN)

print("Authenticated with ngrok.")

## 5. Download the Model

This downloads the model into the Hugging Face cache. vLLM will use this cached copy when it starts.

In [ ]:
from huggingface_hub import snapshot_download

model_cache_path = snapshot_download(repo_id=MODEL_NAME)
print("Model downloaded/cached at:", model_cache_path)

## 6. Start vLLM on Local Port 8000

This starts vLLM in the background. Server logs are written to `/content/vllm.log`.

In [ ]:
import os
import signal
import subprocess
import time

import requests

try:
    if VLLM_PROCESS.poll() is None:
        print("Stopping previous vLLM process...")
        VLLM_PROCESS.terminate()
        VLLM_PROCESS.wait(timeout=20)
except NameError:
    pass

log_path = "/content/vllm.log"
log_file = open(log_path, "w")

cmd = [
    "vllm", "serve", MODEL_NAME,
    "--host", "0.0.0.0",
    "--port", str(PORT),
    "--dtype", "float16",
    "--max-model-len", str(MAX_MODEL_LEN),
    "--gpu-memory-utilization", str(GPU_MEMORY_UTILIZATION),
    "--served-model-name", MODEL_NAME,
    "--api-key", API_KEY,
]

print("Starting vLLM:")
print(" ".join(cmd))

VLLM_PROCESS = subprocess.Popen(
    cmd,
    stdout=log_file,
    stderr=subprocess.STDOUT,
    env=os.environ.copy(),
)

headers = {"Authorization": f"Bearer {API_KEY}"}
local_models_url = f"http://127.0.0.1:{PORT}/v1/models"

def print_log_tail(path: str, n: int = 80) -> None:
    try:
        with open(path, "r", errors="replace") as f:
            lines = f.readlines()[-n:]
        print("".join(lines))
    except FileNotFoundError:
        print(f"Log file not found: {path}")

print("Waiting for vLLM to become ready. This can take several minutes on first load...")
deadline = time.time() + 20 * 60
last_error = None

while time.time() < deadline:
    if VLLM_PROCESS.poll() is not None:
        print("vLLM process exited early. Last log lines:")
        print_log_tail(log_path)
        raise RuntimeError("vLLM failed to start.")
    try:
        response = requests.get(local_models_url, headers=headers, timeout=5)
        if response.ok:
            print("vLLM is ready:", response.json())
            break
        last_error = f"HTTP {response.status_code}: {response.text[:200]}"
    except Exception as exc:
        last_error = repr(exc)
    time.sleep(10)
else:
    print("Timed out waiting for vLLM. Last error:", last_error)
    print("Last log lines:")
    print_log_tail(log_path)
    raise TimeoutError("vLLM did not become ready within 20 minutes.")

## 7. Expose Port 8000 with ngrok

In [ ]:
from pyngrok import ngrok

ngrok.kill()
tunnel = ngrok.connect(PORT, "http")
PUBLIC_BASE_URL = tunnel.public_url.rstrip("/")
PUBLIC_V1_URL = f"{PUBLIC_BASE_URL}/v1"

print("Public ngrok base URL:", PUBLIC_BASE_URL)
print("Public OpenAI-compatible vLLM URL:", PUBLIC_V1_URL)
print()
print("Use these environment variables for the Legal AI Assistant backend:")
print(f"export LLM_API_URL={PUBLIC_V1_URL}")
print(f"export MODEL_NAME={MODEL_NAME}")
print(f"export LLM_API_KEY={API_KEY}")

## 8. Test the Public vLLM Endpoint

In [ ]:
import json
import requests

headers = {
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json",
}

models_response = requests.get(f"{PUBLIC_V1_URL}/models", headers=headers, timeout=30)
print("/v1/models status:", models_response.status_code)
print(models_response.text[:1000])

payload = {
    "model": MODEL_NAME,
    "messages": [
        {"role": "system", "content": "You are a concise assistant."},
        {"role": "user", "content": "Say hello in one sentence."},
    ],
    "max_tokens": 64,
    "temperature": 0.1,
}

chat_response = requests.post(
    f"{PUBLIC_V1_URL}/chat/completions",
    headers=headers,
    json=payload,
    timeout=120,
)
print("/v1/chat/completions status:", chat_response.status_code)
print(json.dumps(chat_response.json(), indent=2)[:3000])

## 9. Connect the Legal AI Assistant Backend

In your project terminal, use the printed values from step 7:

```bash
export LLM_API_URL=https://YOUR-NGROK-URL/v1
export MODEL_NAME=Qwen/Qwen2.5-3B-Instruct
export LLM_API_KEY=dummy-key
bash scripts/run.sh
```

Then upload a document and ask a question through the Streamlit UI or FastAPI `/query` endpoint.

## 10. Stop Server and Tunnel

Run this when you are done. Colab runtimes are temporary; the ngrok URL changes when you restart the tunnel.

In [ ]:
from pyngrok import ngrok

ngrok.kill()

try:
    if VLLM_PROCESS.poll() is None:
        VLLM_PROCESS.terminate()
        VLLM_PROCESS.wait(timeout=20)
        print("Stopped vLLM process.")
    else:
        print("vLLM process was already stopped.")
except NameError:
    print("No vLLM process variable found.")

print("Stopped ngrok tunnels.")